# 76 Unity Catalog · Row Filter y Column Mask

Vas a aplicar seguridad de grano fino sobre PII del padrón con una sola identidad. `is_account_group_member('grupo-inexistente')` devuelve `FALSE` y reproduce la experiencia no autorizada; luego vas a autorizar explícitamente al usuario actual y comparar ambas vistas.

In [ ]:
CATALOG = "big_data_ii_2025"
SCHEMA  = "spark_examples"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(f"Trabajando en {CATALOG}.{SCHEMA}")

In [ ]:
# Verifica la tabla protegida y captura la identidad que evaluarán las políticas.
TABLA_PADRON = f"{CATALOG}.{SCHEMA}.padron"
assert spark.catalog.tableExists(TABLA_PADRON), (
    f"No existe {TABLA_PADRON}. Ejecutá primero el notebook 75_UC_Jerarquia_y_Permisos.")
usuario = spark.sql("SELECT current_user() AS usuario").first()["usuario"]
print(f"Identidad de la demostración: {usuario}")

## Línea base sin protección

Observá algunos valores antes de aplicar políticas. No copies ni exportés estas filas: contienen identificadores directos de personas.

In [ ]:
# Obtiene una línea base acotada para comparar el efecto de las políticas posteriores.
display(spark.sql(f"""
SELECT cedula, nombre_completo, provincia, canton
FROM {TABLA_PADRON}
LIMIT 10
"""))

## Column mask · perspectiva no autorizada

La función devuelve la cédula completa solo a integrantes de `oficiales_padron`. Como ese grupo no existe en Free Edition, vas a observar el resultado enmascarado. Antes de aplicar la política quitamos cualquier máscara anterior para que *Restart & Run All* sea seguro.

In [ ]:
FUNC_MASK = f"{CATALOG}.{SCHEMA}.mask_cedula"
# Retira una máscara previa para que CREATE OR REPLACE y SET MASK sean reejecutables.
try:
    spark.sql(f"ALTER TABLE {TABLA_PADRON} ALTER COLUMN cedula DROP MASK")
except Exception as e:
    print(f"No había una máscara previa que quitar: {e}")

# Devuelve la cédula real al grupo autorizado y solo los últimos cuatro dígitos al resto.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FUNC_MASK}(cedula STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('oficiales_padron') THEN cedula
  ELSE CONCAT('*****', SUBSTRING(cedula, -4))
END
""")
# Vincula el UDF a la columna; los datos Delta subyacentes no se reescriben.
spark.sql(f"ALTER TABLE {TABLA_PADRON} ALTER COLUMN cedula SET MASK {FUNC_MASK}")

# Repite la consulta base para hacer visible la perspectiva no autorizada.
display(spark.sql(f"""
SELECT cedula, nombre_completo, provincia, canton
FROM {TABLA_PADRON}
LIMIT 10
"""))

## Column mask · perspectiva autorizada

Ahora la misma función reconoce explícitamente la identidad actual. Así podés verificar la rama autorizada sin crear usuarios ni grupos adicionales.

In [ ]:
# Escapa la identidad antes de interpolarla como literal SQL.
usuario_sql = usuario.replace("'", "''")
# Amplía temporalmente la política para simular la perspectiva de una persona autorizada.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FUNC_MASK}(cedula STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('oficiales_padron')
       OR current_user() = '{usuario_sql}' THEN cedula
  ELSE CONCAT('*****', SUBSTRING(cedula, -4))
END
""")

# La misma consulta ahora devuelve el valor real por la condición de current_user().
display(spark.sql(f"""
SELECT cedula, nombre_completo, provincia, canton
FROM {TABLA_PADRON}
LIMIT 10
"""))

El dato almacenado **nunca cambió**. Unity Catalog evalúa la máscara en tiempo de consulta y transforma el resultado según la identidad que ejecuta el `SELECT`.

## Row filter por provincia

El filtro representa a un analista limitado a Heredia. Prestá atención al comportamiento: la consulta no advierte que ocultó filas; `COUNT(*)` simplemente devuelve un número menor.

In [ ]:
FUNC_FILTRO = f"{CATALOG}.{SCHEMA}.filtro_provincia"
# Limpia una política anterior antes de redefinir el row filter.
try:
    spark.sql(f"ALTER TABLE {TABLA_PADRON} DROP ROW FILTER")
except Exception as e:
    print(f"No había un row filter previo que quitar: {e}")

# Autoriza todo el país al grupo nacional y restringe las demás identidades a Heredia.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FUNC_FILTRO}(provincia STRING)
RETURNS BOOLEAN
RETURN is_account_group_member('padron_nacional')
    OR provincia = 'Heredia'
""")
# Aplica el UDF a provincia para evaluar cada fila durante la consulta.
spark.sql(f"ALTER TABLE {TABLA_PADRON} SET ROW FILTER {FUNC_FILTRO} ON (provincia)")

# Comprueba el efecto silencioso mediante el total visible y las provincias accesibles.
display(spark.sql(f"SELECT COUNT(*) AS filas_visibles FROM {TABLA_PADRON}"))
display(spark.sql(f"SELECT DISTINCT provincia FROM {TABLA_PADRON} ORDER BY provincia"))

## Máscara y filtro acumulados

Para hacer visibles ambos controles a la vez, volvemos a la versión no autorizada de la máscara y mantenemos el row filter. Esperás ver únicamente Heredia y cédulas parciales.

In [ ]:
# Restablece la máscara no autorizada mientras el row filter permanece activo.
spark.sql(f"""
CREATE OR REPLACE FUNCTION {FUNC_MASK}(cedula STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('oficiales_padron') THEN cedula
  ELSE CONCAT('*****', SUBSTRING(cedula, -4))
END
""")

# Verifica que las filas filtradas y las cédulas enmascaradas se acumulen en el mismo SELECT.
display(spark.sql(f"""
SELECT cedula, nombre_completo, provincia, canton
FROM {TABLA_PADRON}
LIMIT 10
"""))

# Crea una vista sobre la tabla protegida para demostrar que las políticas se propagan.
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.v_padron_seguro AS
SELECT cedula, nombre_completo, provincia, canton, distrito
FROM {TABLA_PADRON}
""")
# Agrupa desde la vista; el resultado sigue limitado por el row filter de la tabla base.
display(spark.sql(f"""
SELECT provincia, COUNT(*) AS filas_visibles
FROM {CATALOG}.{SCHEMA}.v_padron_seguro
GROUP BY provincia
"""))

La view sigue protegida porque las políticas se evalúan sobre la tabla base. Un consumidor no puede eludirlas creando otra consulta o agregación encima.

## Limitaciones y evolución hacia ABAC

- Las tablas con row filters o column masks no admiten `CLONE` ni consultas de time travel.
- Estas políticas no están soportadas en Delta Sharing estándar.
- El UDF se evalúa en cada consulta: mantenelo simple y determinista.
- Las cuotas son 10.000 políticas por metastore, 100 por catálogo o esquema y 50 por tabla.
- **ABAC** centraliza políticas mediante *governed tags*, en lugar de repetir `ALTER TABLE` objeto por objeto.

## Limpieza

Quitamos ambas políticas para que el notebook 77 pueda agregar todo el padrón. Las funciones permanecen como objetos gobernados que podés inspeccionar.

In [ ]:
# Retira ambas políticas de forma independiente para no ocultar un fallo de limpieza.
try:
    spark.sql(f"ALTER TABLE {TABLA_PADRON} ALTER COLUMN cedula DROP MASK")
except Exception as e:
    print(f"No se pudo quitar la máscara: {e}")
try:
    spark.sql(f"ALTER TABLE {TABLA_PADRON} DROP ROW FILTER")
except Exception as e:
    print(f"No se pudo quitar el row filter: {e}")
# Deja la tabla sin restricciones para que el agregado del notebook 77 use el padrón completo.
print("Políticas retiradas; padron quedó disponible para el notebook 77.")

## Cierre

- Comparaste las perspectivas no autorizada y autorizada de una column mask.
- Verificaste que una máscara transforma resultados, no los datos almacenados.
- Aplicaste un row filter y observaste su efecto silencioso sobre `COUNT(*)`.
- Confirmaste que máscara, filtro y view acumulan las políticas de la tabla base.
- Limpiaste las políticas para continuar con lineage.